# 4회차 · 비지도학습 — 정답 없는 데이터에서 무리(군집) 찾기
**2026 경기도 공유학교 · 인공지능 기초 (김순찬 교수)** · 성취기준 [인일 03-02]

## 오늘의 학습 목표
1. 비지도학습이 지도학습과 어떻게 다른지 설명할 수 있다.
2. **k-means**로 데이터를 군집으로 나누고, **엘보 곡선·실루엣 점수**로 군집 개수 k를 정할 수 있다.
3. **PCA**로 여러 변수를 2차원 그림으로 줄여 볼 수 있다.
4. 데이터 모양(둥근 덩어리 / 이어진 모양 / 계층)에 맞는 기법(k-means · DBSCAN · 계층적 군집)을 고를 수 있다.

## 사용법
- 셀을 **위에서부터 순서대로** 실행하세요. 셀을 클릭하고 **Shift + Enter**.
- 🧩 표시 셀은 직접 코드를 채우는 셀입니다. 막히면 바로 아래 **정답 보기**를 펼치세요.
- 오류가 나면 *런타임 → 런타임 다시 시작* 후 처음부터 다시 실행하면 대부분 해결됩니다.

## STEP 0. 준비 — 도구 불러오기와 한글 폰트
파이썬에서 **numpy**(숫자 계산), **pandas**(표), **matplotlib**(그림), **scikit-learn**(머신러닝)을 불러옵니다.
Colab의 그림은 기본적으로 한글이 깨지므로 나눔 폰트를 설치해 둡니다. (10초쯤 걸립니다. "설정 완료"가 나오면 성공.)

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
import subprocess, sys
try:
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], check=False, capture_output=True)   # Colab 한글 폰트
    import matplotlib.font_manager as fm; fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'); plt.rcParams['font.family']='NanumGothic'
except Exception:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'   # 로컬 검증용
plt.rcParams['axes.unicode_minus'] = False
print("설정 완료! numpy", np.__version__, "/ pandas", pd.__version__)

## STEP 1. 정답 없는 점들을 만들어 보기 — `make_blobs`
`make_blobs`는 연습용으로 **몇 개의 덩어리(blob)** 모양 점들을 만들어 주는 함수입니다.
사실 이 함수는 각 점이 어느 덩어리에서 나왔는지(정답 `y`)도 같이 주지만, **비지도학습은 정답을 쓰지 않으므로** 우리는 `X`(점의 좌표)만 사용합니다.
- `n_samples=300` : 점 300개
- `centers=4` : 덩어리 4개
- `cluster_std=1.0` : 덩어리가 퍼진 정도
- `random_state=42` : 실행할 때마다 같은 점이 나오도록 고정

그림을 보고 **"덩어리가 몇 개로 보이는가?"** 를 먼저 눈으로 세어 보세요.

In [ ]:
from sklearn.datasets import make_blobs

X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)
print("X의 모양(점 개수, 특징 개수):", X.shape)
print("앞의 점 3개:\n", X[:3].round(2))

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], s=15, color='gray')      # 색 없이 회색 점만 — 정답을 모른다고 가정
plt.title('정답 없는 데이터: 덩어리가 몇 개로 보이나요?'); plt.xlabel('특징 1'); plt.ylabel('특징 2')
plt.show()

## STEP 2. k-means로 군집 나누기
**k-means**는 "군집 개수 k를 정해 주면, 중심 k개를 놓고 ① 가장 가까운 중심에 점을 배정 → ② 중심을 배정된 점들의 평균으로 이동" 을 반복하는 알고리즘입니다.
- `KMeans(n_clusters=4)` : 군집 4개로 나누겠다
- `n_init=10` : 처음 중심을 10번 다르게 놓아 보고 가장 좋은 결과를 고른다 (초기값 운에 덜 흔들림)
- `.fit_predict(X)` : 학습하고, 각 점의 군집 번호(0~3)를 돌려준다
- `.cluster_centers_` : 최종 중심 좌표

출력의 **군집 번호는 순서에 의미가 없습니다** (0번이 특별히 좋은 게 아님). 그림에서 색이 잘 나뉘었는지, 검은 X(중심)가 각 덩어리 한가운데 있는지 확인하세요.

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=0)
labels = km.fit_predict(X)             # 각 점의 군집 번호
centers = km.cluster_centers_          # 군집 중심 좌표

print("군집 번호(앞 10개):", labels[:10])
print("군집별 점 개수:", np.bincount(labels))
print("중심 좌표:\n", centers.round(2))

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=15)                  # 군집별 색
plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=200, c='black', label='중심')
plt.legend(); plt.title('k-means (k=4) 결과'); plt.show()

## STEP 3. k를 바꾸면 어떻게 될까?
같은 데이터를 k=2, 3, 4, 6으로 나눠 봅니다. **k-means는 무조건 k개로 나눕니다** — 덩어리가 4개인데 k=2면 두 덩어리를 억지로 합치고, k=6이면 한 덩어리를 쪼갭니다.
그래서 "k를 어떻게 정할까?"가 다음 질문이 됩니다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3))
for ax, k in zip(axes, [2, 3, 4, 6]):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=lab, cmap='tab10', s=10)
    ax.set_title(f'k = {k}'); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('k를 바꾸면 나누는 방식이 달라진다'); plt.show()

## STEP 4. 엘보(팔꿈치) 곡선으로 k 고르기
`km.inertia_`는 **각 점에서 자기 군집 중심까지 거리의 제곱을 모두 더한 값**입니다. 군집이 촘촘할수록 작습니다.
k를 늘리면 이 값은 항상 줄어들지만, "진짜 덩어리 수"를 넘어가면 **조금밖에 안 줄어듭니다.** 그래프가 팔꿈치처럼 **확 꺾이는 지점**이 후보 k입니다.

In [ ]:
ks = range(1, 10)
inertias = []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    inertias.append(km_k.inertia_)
    print(f"k={k}: inertia = {km_k.inertia_:,.0f}")

plt.figure(figsize=(5.5, 3.5))
plt.plot(ks, inertias, 'o-')
plt.xlabel('군집 개수 k'); plt.ylabel('inertia (중심까지 거리 제곱 합)'); plt.title('엘보 곡선'); plt.show()

## STEP 5. 실루엣 점수로 k 고르기
**실루엣 점수**는 각 점에 대해 "내 군집 친구들과는 가깝고, 다른 군집과는 먼가?"를 −1~1로 매긴 뒤 평균한 값입니다.
- 1에 가까움 = 군집이 뚜렷하게 나뉨 / 0 근처 = 경계가 애매함 / 음수 = 잘못 배정된 점이 많음
- k=1은 계산할 수 없어 k=2부터 봅니다. **가장 높은 k**를 고릅니다. (엘보 곡선의 결론과 같은지 비교해 보세요.)

In [ ]:
from sklearn.metrics import silhouette_score

scores = {}
for k in range(2, 9):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    scores[k] = silhouette_score(X, lab)
    print(f"k={k}: 실루엣 점수 = {scores[k]:.3f}")

best_k = max(scores, key=scores.get)
print("\n실루엣 점수가 가장 높은 k =", best_k)

plt.figure(figsize=(5.5, 3.5))
plt.bar(list(scores.keys()), list(scores.values()))
plt.xlabel('군집 개수 k'); plt.ylabel('실루엣 점수'); plt.title('실루엣 점수 (높을수록 좋음)'); plt.show()

## STEP 6. 진짜 같은 데이터 — '학생 생활 유형' 데이터 만들기
이번에는 우리 반 같은 **학생 150명의 하루 생활 시간** 데이터입니다 (수면·공부·게임·운동·SNS·독서, 단위: 시간).
이 데이터에는 **유형 이름(정답)이 없습니다.** "비슷한 생활 패턴을 가진 학생끼리 묶으면 어떤 유형이 나올까?" — 이것이 오늘의 진짜 문제입니다.

> 데이터는 코드가 직접 만듭니다(고정 seed라 누가 실행해도 같은 값). 선생님이 나눠 준 `student_types.csv`와 같은 내용입니다.

In [ ]:
def make_student_types(n_per=50, seed=2026):
    rng = np.random.default_rng(seed)
    centers = {                      # 숨겨진 유형 3개의 '평균 생활' (학생들은 이 이름을 모름)
        'A': [6.5, 5.0, 0.7, 0.5, 1.0, 1.2],
        'B': [6.0, 1.5, 4.0, 0.3, 1.5, 0.3],
        'C': [7.5, 2.0, 1.0, 2.5, 3.0, 0.5],
    }
    sd = np.array([0.7, 0.8, 0.7, 0.5, 0.7, 0.4])
    rows = [np.clip(rng.normal(c, sd, size=(n_per, 6)), 0, None) for c in centers.values()]
    cols = ['수면시간', '공부시간', '게임시간', '운동시간', 'SNS시간', '독서시간']
    df = pd.DataFrame(np.round(np.vstack(rows), 1), columns=cols)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)   # 섞기

students = make_student_types()
print("학생 수, 변수 수:", students.shape)
display(students.head())
display(students.describe().round(2))

## STEP 7. 표준화 — 단위가 다른 변수를 같은 저울에 올리기
k-means는 **거리**로 판단합니다. 그런데 수면시간은 6~8 사이에서 움직이고 독서시간은 0~2 사이에서 움직이면, 거리 계산에서 수면시간이 훨씬 큰 목소리를 냅니다.
`StandardScaler`는 각 변수를 **(값 − 평균) ÷ 표준편차**로 바꿔, 모든 변수가 평균 0·표준편차 1이 되게 합니다. 이렇게 하면 어떤 변수도 단위 때문에 유리하지 않습니다.
표준화 후 각 열의 평균이 0(아주 작은 수), 표준편차가 1인지 확인하세요.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xs = scaler.fit_transform(students)          # 표준화된 값 (numpy 배열)
print("표준화 후 열별 평균:", Xs.mean(axis=0).round(3))
print("표준화 후 열별 표준편차:", Xs.std(axis=0).round(3))

## STEP 8. 학생 데이터에 k-means 적용 + k 정하기
STEP 4·5에서 배운 두 방법을 학생 데이터에 그대로 적용합니다. 6개 변수라 그림으로 바로 볼 수는 없지만, 엘보와 실루엣은 **몇 차원이든 계산할 수 있습니다.**
두 방법이 가리키는 k를 확인한 뒤, 그 k로 최종 군집을 만듭니다.

In [ ]:
inertias, sils = {}, {}
for k in range(2, 8):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xs)
    inertias[k] = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs).inertia_
    sils[k] = silhouette_score(Xs, lab)
    print(f"k={k}: inertia={inertias[k]:.1f}, 실루엣={sils[k]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(list(inertias), list(inertias.values()), 'o-'); ax[0].set_title('엘보 곡선'); ax[0].set_xlabel('k')
ax[1].bar(list(sils), list(sils.values())); ax[1].set_title('실루엣 점수'); ax[1].set_xlabel('k')
plt.show()

best_k = max(sils, key=sils.get)
print("→ 실루엣이 가장 높은 k =", best_k)
km_s = KMeans(n_clusters=best_k, n_init=10, random_state=0)
students['군집'] = km_s.fit_predict(Xs)
print(students['군집'].value_counts().sort_index())

## STEP 9. 군집 해석 — 각 군집은 어떤 학생들인가?
군집 번호(0, 1, 2)만으로는 아무것도 알 수 없습니다. **군집별 평균**을 표로 보면 각 군집의 특징이 드러납니다.
`groupby('군집').mean()`은 군집별로 각 변수의 평균을 계산합니다. 표를 보고 **"공부시간이 가장 긴 군집", "게임시간이 가장 긴 군집"** 처럼 특징을 찾고 이름을 붙여 보세요.
아래 막대그래프는 같은 표를 그림으로 그린 것입니다.

In [ ]:
profile = students.groupby('군집').mean().round(2)
display(profile)

profile.T.plot(kind='bar', figsize=(8, 3.6))
plt.title('군집별 평균 생활 시간'); plt.ylabel('시간'); plt.xticks(rotation=0); plt.legend(title='군집'); plt.show()

# 각 변수에서 평균이 가장 큰 군집을 자동으로 찾아 힌트로 출력
for col in profile.columns:
    print(f"{col}: 가장 긴 군집 = {profile[col].idxmax()} ({profile[col].max()}시간)")

## 🧩 직접 해보기 1 — 군집에 이름 붙이기
STEP 9의 표를 보고 각 군집 번호에 어울리는 이름을 지어 `names` 딕셔너리를 채우세요. (예: '공부 집중형', '게임 몰입형', '활동·SNS형' …)
힌트: 실행 결과에서 군집 번호가 여러분의 화면과 같은지 반드시 확인하세요. 번호는 실행 환경에 따라 바뀔 수 있습니다.

In [ ]:
# TODO: 군집 번호 → 이름  (STEP 9의 표를 보고 채우세요)
names = {0: '이름을 지어 보세요', 1: '이름을 지어 보세요', 2: '이름을 지어 보세요'}

students['유형이름'] = students['군집'].map(names)
print(students['유형이름'].value_counts())
display(students.head())

<details><summary>정답 예시 보기</summary>

```python
# 군집별 평균표에서 공부시간이 가장 긴 군집·게임시간이 가장 긴 군집·운동/SNS가 긴 군집을 찾은 뒤 (번호는 환경마다 다를 수 있음)
names = {profile['공부시간'].idxmax(): '공부 집중형',
         profile['게임시간'].idxmax(): '게임 몰입형',
         profile['운동시간'].idxmax(): '활동·SNS형'}
```
</details>

## STEP 10. PCA로 6차원을 2차원 그림으로 보기
6개 변수를 한 번에 산점도로 그릴 수는 없습니다. **PCA(주성분 분석)**는 데이터가 **가장 넓게 퍼진 방향**부터 새 축(PC1, PC2, …)을 잡아 주는 방법입니다. 앞의 두 축만 쓰면 정보를 최대한 살린 2차원 그림이 됩니다.
- `explained_variance_ratio_` : 각 축이 전체 퍼짐(정보)의 몇 %를 담는지. PC1+PC2가 60% 이상이면 2차원 그림을 꽤 믿을 만합니다.
- 군집 색이 2차원 그림에서도 잘 나뉘어 보이는지 확인하세요.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
Z = pca.fit_transform(Xs)                     # 표준화된 6차원 → 2차원
ratio = pca.explained_variance_ratio_
print("PC1, PC2가 담는 정보 비율:", ratio.round(3), "→ 합계", ratio.sum().round(3))

plt.figure(figsize=(5.5, 4.2))
for c in sorted(students['군집'].unique()):
    m = students['군집'] == c
    plt.scatter(Z[m, 0], Z[m, 1], s=18, label=f"군집 {c}: {students.loc[m, '유형이름'].iloc[0]}")
plt.xlabel(f'PC1 ({ratio[0]*100:.0f}%)'); plt.ylabel(f'PC2 ({ratio[1]*100:.0f}%)')
plt.legend(fontsize=8); plt.title('PCA 2차원 + k-means 군집'); plt.show()

# 각 PC가 어떤 변수로 만들어졌는지 (절댓값이 클수록 그 변수의 영향이 큼)
display(pd.DataFrame(pca.components_, columns=students.columns[:6], index=['PC1', 'PC2']).round(2))

## STEP 11. 계층적 군집 — 가까운 것끼리 차례로 합치기
**계층적 군집**은 k를 미리 정하지 않습니다. 가장 가까운 두 점(또는 무리)을 합치는 일을 하나가 될 때까지 반복하고, 그 과정을 **덴드로그램(나무 그림)** 으로 그립니다.
나무를 어느 높이에서 **가로로 자르느냐**에 따라 군집 개수가 정해집니다. (학생 30명만 뽑아 이름표가 보이게 그립니다.)
`method='ward'`는 "합쳤을 때 퍼짐이 가장 적게 늘어나는 쌍"을 먼저 합치는 방식입니다.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sample = students.sample(30, random_state=1)
Zs = linkage(scaler.transform(sample.iloc[:, :6]), method='ward')

plt.figure(figsize=(11, 3.8))
dendrogram(Zs, labels=[f'학생{i}' for i in sample.index], leaf_font_size=8)
plt.axhline(8, ls='--', color='red'); plt.text(2, 8.4, '이 높이에서 자르면 → 3개 군집', color='red')
plt.ylabel('합쳐질 때의 거리'); plt.title('계층적 군집 덴드로그램 (학생 30명)'); plt.show()

cut = fcluster(Zs, t=3, criterion='maxclust')     # 3개 군집으로 자르기
print("자른 결과(군집 번호 1~3):", cut)
print("k-means 결과와 비교:", sample['군집'].values)

## STEP 12. k-means가 실패하는 모양 — 초승달 데이터와 DBSCAN
k-means는 **둥근 덩어리**를 가정합니다. 초승달 두 개가 얽힌 모양(`make_moons`)에서는 엉뚱하게 자릅니다.
**DBSCAN**은 "반지름 `eps` 안에 이웃이 `min_samples`개 이상이면 같은 무리"라는 규칙으로 **촘촘히 이어진 모양**을 따라갑니다. k를 정할 필요가 없고, 어디에도 속하지 않는 점은 **−1(잡음)** 로 표시합니다.
`eps`를 0.1, 0.3으로 바꿔 보며 결과가 어떻게 변하는지도 관찰하세요.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

Xm, _ = make_moons(n_samples=300, noise=0.07, random_state=0)
lab_km = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xm)
lab_db = DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm)
print("DBSCAN이 찾은 군집 번호:", np.unique(lab_db), "(−1은 잡음)")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].scatter(Xm[:, 0], Xm[:, 1], c=lab_km, cmap='tab10', s=14); ax[0].set_title('k-means (k=2): 잘못 자름')
ax[1].scatter(Xm[:, 0], Xm[:, 1], c=lab_db, cmap='tab10', s=14); ax[1].set_title('DBSCAN (eps=0.2): 모양대로 나눔')
plt.show()

## 🧩 직접 해보기 2 — 어떤 기법을 고를까?
아래 세 가지 상황에 맞는 기법을 `choice`에 적고, 이유를 한 줄씩 쓰세요. 후보: `'k-means'`, `'DBSCAN'`, `'계층적 군집'`, `'PCA'`
1. 온라인 게임 회원 10만 명을 비슷한 플레이 습관끼리 **5개 그룹**으로 나눠 이벤트를 보내고 싶다.
2. 지도 위 배달 주문 위치가 **길을 따라 길게 이어진 모양**으로 몰려 있다. 몰린 구역을 찾고 싶다.
3. 설문 문항 **40개**로 응답자를 조사했는데, 한눈에 보이는 **그림 한 장**으로 요약하고 싶다.

In [ ]:
# TODO: 상황별로 알맞은 기법과 이유를 채우세요
choice = {
    1: ('여기에 기법', '이유: '),
    2: ('여기에 기법', '이유: '),
    3: ('여기에 기법', '이유: '),
}
for i, (m, why) in choice.items():
    print(f"상황 {i}: {m}  —  {why}")

<details><summary>정답 예시 보기</summary>

1. **k-means** — 그룹 수(5)가 정해져 있고 데이터가 많아 빠른 방법이 필요함.
2. **DBSCAN** — 둥근 덩어리가 아니라 길게 이어진 모양이고, 몰린 정도(밀도)로 구역을 찾는 문제.
3. **PCA** — 군집이 아니라 **차원 축소**(40개 → 2개 축) 문제.
</details>

## 📝 오늘의 결과물 제출
아래 셀에 이름·학번을 쓰고, **군집 이름 3개 + 각 군집의 특징 한 줄 + 오늘 배운 기법 중 하나를 고른 이유**를 적은 뒤 실행하세요. 출력 화면을 캡처해 제출합니다.

**자기 점검**
- [ ] k-means가 '중심 이동 반복'으로 군집을 찾는다는 것을 설명할 수 있다.
- [ ] 엘보 곡선과 실루엣 점수로 k를 정해 보았다.
- [ ] 표준화를 왜 하는지 말할 수 있다.
- [ ] 군집별 평균표로 군집에 이름을 붙였다.
- [ ] 초승달 모양에서 k-means 대신 DBSCAN을 쓰는 이유를 안다.

In [ ]:
이름 = "홍길동"          # ← 수정
학번 = "10101"           # ← 수정
군집_설명 = {
    0: "특징: ",          # ← 군집 0의 특징 한 줄
    1: "특징: ",
    2: "특징: ",
}
기법_선택_이유 = "나는 ___ 데이터에 ___ 기법을 쓰겠다. 왜냐하면 ___"

print("=" * 50)
print(f"4회차 비지도학습 결과물  |  {이름} ({학번})")
print("=" * 50)
print("선택한 k =", best_k, "| 실루엣 점수 =", round(sils[best_k], 3))
for c in sorted(names):
    print(f"군집 {c} [{names[c]}] {군집_설명.get(c, '')}")
print("기법 선택:", 기법_선택_이유)
display(profile)